> **Versión corregida (v2.12, ejecución verificada)** — generada a partir de `notebooks/06_q_learning_capacidad_v2.ipynb` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de configuración del repositorio sustituida por el bootstrap portable (`PROJECT_ROOT`/`REPO_ROOT`).
> - Se eliminan `from google.colab import drive`, `drive.mount(...)` y las rutas fijas `/content/drive/MyDrive/PIDA-RL-Diabetes`; resultados en `RESULTS_V2_DIR` (`results/capacidad_v2`, o `results/capacidad_v2_prueba` en modo prueba).
> - Modo prueba: submuestra estratificada por `categoria_riesgo` (sin crear columnas) y capacidad escalada proporcionalmente para conservar la presión de recursos por persona.
> - Modo prueba: `EPISODIOS_ENTRENAMIENTO = 5` (configurable con `PIDA_EPISODIOS_Q`).
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña y pocos episodios).


# 06 — Q-learning con capacidad operativa V2

Entrena Q-learning tabular y lo compara con las políticas Aleatoria, Reglas simples y Reglas escalonadas. La cohorte mantiene únicamente riesgo **bajo, medio y alto**; no se crea una cuarta categoría.

> Es un experimento de simulación metodológica, no una recomendación clínica.


In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'main')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

for _mod, _pip in [('gymnasium', 'gymnasium'), ('pandas', 'pandas')]:
    if importlib.util.find_spec(_mod) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', _pip])
try:
    print('Commit evaluado:', subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip())
except Exception as exc:  # copia sin .git
    print('Commit evaluado: no disponible', exc)


## Configuración

Se separan las semillas de entrenamiento y evaluación para reducir fuga experimental. Los hiperparámetros son explícitos y ajustables.


In [ ]:
CAPACIDAD_MENSUAL = 100
COSTOS_ACCION = {0: 0, 1: 1, 2: 3, 3: 5}
HORIZONTE_MESES = 12
EPISODIOS_ENTRENAMIENTO = 1500
SEMILLA_ENTRENAMIENTO = 10000
SEMILLAS_EVALUACION = (42, 123, 2026)
ALPHA = 0.10
GAMMA = 0.95
EPSILON_INICIAL = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.995
# V2.16: GAMMA se aplica por mes (γ^t, t = mes), como en la Definición del PIDA.
# Dentro de un mes las decisiones no se descuentan entre sí. 'decision' reproduce V2.15.
DESCUENTO = 'mensual'
ORDEN_ALEATORIO = True  # el orden de atención se baraja cada mes con la semilla del episodio

if MODO_PRUEBA:
    EPISODIOS_ENTRENAMIENTO = int(os.environ.get('PIDA_EPISODIOS_Q', '5'))
print('Episodios de entrenamiento:', EPISODIOS_ENTRENAMIENTO)


## Cohorte pública procesada

Carga el mismo archivo producido por el notebook 03; no genera pacientes nuevos.


In [ ]:
import pandas as pd
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'
OUTPUT_DIR = RESULTS_V2_DIR / 'q_learning'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cohorte = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})

# Modo prueba: cohorte pequeña estratificada y capacidad proporcional.
N_COHORTE_COMPLETA = len(cohorte)
cohorte, FACTOR_COHORTE = muestra_cohorte_prueba(cohorte)
CAPACIDAD_MENSUAL = escalar_capacidad(CAPACIDAD_MENSUAL, FACTOR_COHORTE)
assert set(cohorte['categoria_riesgo'].unique()) <= {'bajo', 'medio', 'alto'}
print(f'Cohorte usada: {len(cohorte)} de {N_COHORTE_COMPLETA} perfiles | capacidad mensual: {CAPACIDAD_MENSUAL}')
print(cohorte['categoria_riesgo'].value_counts().to_dict())
assert set(cohorte['categoria_riesgo'].unique()) <= {'bajo', 'medio', 'alto'}
print(cohorte.shape, cohorte['categoria_riesgo'].value_counts().to_dict())


## Entrenamiento

La tabla usa riesgo ordinal, `score_riesgo`, tiempo sin contacto, mes y recursos restantes. La capacidad forma parte del estado.


In [ ]:
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.policies.q_learning import entrenar_q_learning

def crear_entorno():
    return DiabetesFollowUpEnv(cohorte, CAPACIDAD_MENSUAL, COSTOS_ACCION, HORIZONTE_MESES,
                               orden_aleatorio=ORDEN_ALEATORIO)

agente, historial = entrenar_q_learning(
    crear_entorno, EPISODIOS_ENTRENAMIENTO, SEMILLA_ENTRENAMIENTO,
    ALPHA, GAMMA, EPSILON_INICIAL, EPSILON_MIN, EPSILON_DECAY,
    descuento=DESCUENTO,
)
historial.to_csv(OUTPUT_DIR / 'historial_entrenamiento_q_learning.csv', index=False)
agente.tabla_dataframe().to_csv(OUTPUT_DIR / 'tabla_q.csv', index=False)
agente.guardar(OUTPUT_DIR / 'q_learning_v2.npz')
display(historial.tail())


## Evaluación común

Las cuatro políticas se evalúan en entornos nuevos y con las mismas semillas; Q-learning actúa sin exploración.


In [ ]:
from src.evaluation.evaluate_policies import evaluar_politicas
from src.evaluation.reporting import construir_reporte, exportar_reporte_csv
from src.policies.baselines import POLITICAS_BASE
from src.policies.q_learning import crear_politica_q_learning

politicas = dict(POLITICAS_BASE)
politicas['Q-learning'] = crear_politica_q_learning(agente)
resultados = evaluar_politicas(crear_entorno, politicas, SEMILLAS_EVALUACION)
reporte = construir_reporte(resultados)
exportar_reporte_csv(reporte, OUTPUT_DIR)
detalle = reporte['detalle']
assert set(detalle['categoria_riesgo']) <= {'bajo', 'medio', 'alto'}
assert (detalle['recursos_despues'] >= 0).all()
assert (detalle['recursos_usados_mes'] <= detalle['capacidad_mensual']).all()
display(reporte['resumen_algoritmos'].round(4))
display(reporte['clasificaciones_propuestas'].round(2))
display(reporte['clasificaciones_ejecutadas'].round(2))


## Interpretación responsable

- Compare recompensa, eventos, cobertura, recursos y ajustes por capacidad; no elija una política por una sola métrica.
- Las acciones propuestas pueden diferir de las ejecutadas cuando no hay recursos suficientes.
- Q-learning aprende dentro del simulador y debe reentrenarse si cambian estado, recompensa, costos o capacidad.
- Ningún resultado demuestra efectividad clínica real.
